In [107]:
import pandas as pd
import numpy as np
import re

# =====================================================================
# STEP 1: CALCULATE GROUP STANDINGS (TRACKS 1ST, 2ND, & 3RD PLACE)
# =====================================================================
print("=== STEP 1: PROCESSING REAL GROUP STANDINGS ===")
group_standings = {}

# Award points based on your Group Stage model predictions (Matches 0 to 71)
for idx, row in schedule.loc[0:71].iterrows():
    grp_name = str(row['group']).strip()
    home = row['Home Team']
    away = row['Away Team']
    
    if grp_name not in group_standings:
        group_standings[grp_name] = {}
    if home not in group_standings[grp_name]: group_standings[grp_name][home] = 0
    if away not in group_standings[grp_name]: group_standings[grp_name][away] = 0
    
    # Use the predicted probabilities already stored in your schedule dataframe
    probs = [row['Home Win %'], row['Away Win %'], row['Draw %']]
    highest = probs.index(max(probs))
    
    if highest == 0:    # Home Win
        group_standings[grp_name][home] += 3
    elif highest == 1:  # Away Win
        group_standings[grp_name][away] += 3
    else:               # Draw
        group_standings[grp_name][home] += 1
        group_standings[grp_name][away] += 1

# Build structural mapping keys for Winners, Runners-up, and 3rd-Place teams
group_mapping = {}
for grp_name, teams in group_standings.items():
    sorted_teams = sorted(teams.items(), key=lambda x: x[1], reverse=True)
    
    winner = sorted_teams[0][0] if len(sorted_teams) > 0 else "Brazil"
    runner_up = sorted_teams[1][0] if len(sorted_teams) > 1 else "Argentina"
    third_place = sorted_teams[2][0] if len(sorted_teams) > 2 else "France"
    
    # Clean the string layout keys to guarantee seamless dictionary lookups
    clean_grp = grp_name.lower().replace("-", " ").strip()
    group_mapping[f"{clean_grp} winners"] = winner
    group_mapping[f"{clean_grp} runners up"] = runner_up
    group_mapping[f"{clean_grp} runners-up"] = runner_up
    group_mapping[clean_grp] = third_place  # Bare names (e.g. "Group A") resolve to 3rd place

print("✔ Group standings mapped.")

# =====================================================================
# STEP 2: RUN DYNAMIC KNOCKOUT BRACKET SIMULATION TREE (MATCHES 1-104)
# =====================================================================
print("\n=== STEP 2: RUNNING MACHINE LEARNING SIMULATION LOOP ===")

sim_schedule = schedule.copy()
match_winners = {}
match_losers = {} 

def extract_match_number(team_string):
    match = re.search(r'\d+', team_string)
    return f"Match {match.group()}" if match else None

for idx, row in sim_schedule.iterrows():
    match_label = row['match_number'].strip() 
    home_team = str(row['Home Team']).strip()
    away_team = str(row['Away Team']).strip()
    
    # Overwrite NaN values in 'group' column for knockout matches
    if idx >= 72:
        if 72 <= idx <= 87: stage_name = "Round of 32"
        elif 88 <= idx <= 95: stage_name = "Round of 16"
        elif 96 <= idx <= 99: stage_name = "Quarter-finals"
        elif 100 <= idx <= 101: stage_name = "Semi-finals"
        elif idx == 102: stage_name = "Third Place Playoff"
        elif idx == 103: stage_name = "Final"
        sim_schedule.at[idx, 'group'] = stage_name
    else:
        stage_name = row['group']

    # A. Resolve Initial Round of 32 Placeholders
    home_norm = home_team.lower().replace("-", " ")
    away_norm = away_team.lower().replace("-", " ")
    if home_norm in group_mapping: home_team = group_mapping[home_norm]
    if away_norm in group_mapping: away_team = group_mapping[away_norm]
        
    # B. Resolve Dynamic Bracket Advancement Trees Recursively
    if "winner" in home_team.lower():
        m_num = extract_match_number(home_team)
        home_team = match_winners.get(m_num, f"Winner of {m_num}") 
    elif "runner" in home_team.lower():
        m_num = extract_match_number(home_team)
        home_team = match_losers.get(m_num, f"Loser of {m_num}")

    if "winner" in away_team.lower():
        m_num = extract_match_number(away_team)
        away_team = match_winners.get(m_num, f"Winner of {m_num}")
    elif "runner" in away_team.lower():
        m_num = extract_match_number(away_team)
        away_team = match_losers.get(m_num, f"Loser of {m_num}")

    sim_schedule.at[idx, 'Home Team'] = home_team
    sim_schedule.at[idx, 'Away Team'] = away_team

    # C. Real-Time Stats Verification & Prediction Matrix Extraction
    home_rank, home_rating = team_stats.get(home_team, (25, 1500.0))
    away_rank, away_rating = team_stats.get(away_team, (30, 1480.0))
    
    features_vector = np.array([[
        home_rank, away_rank, home_rating, away_rating, 
        away_rank - home_rank, home_rating - away_rating
    ]])
    
    # Predict probabilities using your trained ML model object
    probs = model.predict_proba(features_vector)[0]
    
    sim_schedule.at[idx, "Home Win %"] = round(probs[0] * 100, 1)
    sim_schedule.at[idx, "Away Win %"] = round(probs[1] * 100, 1)
    sim_schedule.at[idx, "Draw %"] = round(probs[2] * 100, 1)
    
    # D. Set Concrete Match Winner Flags
    if idx >= 72:  # Knockout Rules (No Draws Allowed)
        if probs[0] > probs[1]:
            winner, loser = home_team, away_team
        else:
            winner, loser = away_team, home_team
    else:  # Group Stage Rules
        outcome = np.argmax(probs)
        if outcome == 0: winner, loser = home_team, away_team
        elif outcome == 1: winner, loser = away_team, home_team
        else: winner, loser = "Draw", "None"
        
    sim_schedule.at[idx, "Actual_Winner"] = winner
    match_winners[match_label] = winner
    match_losers[match_label] = loser

print("✔ Dynamic Tournament Simulation Tree Completed Successfully!")

# =====================================================================
# STEP 3: PRINT UNTRUNCATED TEXT SUMMARY OF ALL 104 MATCH PREDICTIONS
# =====================================================================
print("\n" + "="*123)
print(f"{'MATCH':<10} | {'STAGE/GROUP':<20} | {'HOME TEAM':<24} vs {'AWAY TEAM':<24} | {'PROBS (H/A/D)':<20} | {'PREDICTED WINNER'}")
print("="*123)

for idx, row in sim_schedule.iterrows():
    match_num = row['match_number']
    stage = row['group']
    home = row['Home Team']
    away = row['Away Team']
    h_win = row['Home Win %']
    a_win = row['Away Win %']
    draw = row['Draw %']
    winner = row['Actual_Winner']
    
    # Visually separate tournament phases for clean output monitoring
    if idx in [72, 88, 96, 100, 102, 103]:
        print("-" * 123)
        
    if idx >= 72:
        prob_str = f"{h_win}% / {a_win}%"
    else:
        prob_str = f"{h_win}% / {a_win}% / {draw}%"
    
    print(f"{match_num:<10} | {stage:<20} | {home:<24} vs {away:<24} | {prob_str:<20} | {winner}")

print("="*123)

=== STEP 1: PROCESSING REAL GROUP STANDINGS ===
✔ Group standings mapped.

=== STEP 2: RUNNING MACHINE LEARNING SIMULATION LOOP ===
✔ Dynamic Tournament Simulation Tree Completed Successfully!

MATCH      | STAGE/GROUP          | HOME TEAM                vs AWAY TEAM                | PROBS (H/A/D)        | PREDICTED WINNER
Match 1    | Group A              | Mexico                   vs South Africa             | 73.5% / 11.3% / 15.2% | Mexico
Match 2    | Group A              | South Korea              vs Czech Republic           | 55.4% / 21.2% / 23.4% | South Korea
Match 3    | Group B              | Canada                   vs Bosnia and Herzegovina   | 61.0% / 17.3% / 21.7% | Canada
Match 4    | Group D              | United States            vs Paraguay                 | 65.5% / 16.2% / 18.3% | United States
Match 5    | Group C              | Haiti                    vs Scotland                 | 13.7% / 49.1% / 37.2% | Scotland
Match 6    | Group D              | Australia      

In [104]:
import pandas as pd
import numpy as np

# =====================================================================
# STEP 1: INITIALIZE ALL 48 TEAMS & THEIR GRUPS (A to L)
# =====================================================================
groups = {
    'Group A': ['Mexico', 'USA', 'Canada', 'Panama'],
    'Group B': ['Argentina', 'Ecuador', 'Chile', 'Venezuela'],
    'Group C': ['Brazil', 'Uruguay', 'Colombia', 'Paraguay'],
    'Group D': ['France', 'Netherlands', 'Austria', 'Poland'],
    'Group E': ['England', 'Italy', 'Serbia', 'Slovenia'],
    'Group F': ['Belgium', 'Croatia', 'Ukraine', 'Romania'],
    'Group G': ['Spain', 'Portugal', 'Scotland', 'Turkey'],
    'Group L': ['Germany', 'Switzerland', 'Hungary', 'Denmark'] 
    # Note: Expanded tournament structural layout includes groups H, I, J, K as well
}

# Add remaining slots to fully populate a structural mock baseline dataframe
all_teams = []
for g_name, g_teams in groups.items():
    for team in g_teams:
        all_teams.append({'Team': team, 'Group': g_name})

# Fallback filler teams to pad the dataframe array structure up to 48 teams
while len(all_teams) < 48:
    grp_idx = len(all_teams) // 4
    grp_letter = chr(65 + grp_idx) # A, B, C...
    all_teams.append({'Team': f'Team_{len(all_teams)+1}', 'Group': f'Group {grp_letter}'})

teams_df = pd.DataFrame(all_teams)

# =====================================================================
# STEP 2: GENERATE THE FULL 104-MATCH CONFIGURATION DATAFRAME
# =====================================================================
matches_list = []
match_num = 1

# 1. Generate Group Stage (Matches 1 to 72)
for grp, df_grp in teams_df.groupby('Group'):
    grp_teams = df_grp['Team'].tolist()
    # Simple round-robin combinations per group (6 matches per group * 12 groups = 72)
    for i in range(len(grp_teams)):
        for j in range(i + 1, len(grp_teams)):
            if match_num <= 72:
                matches_list.append({
                    'match_number': f"Match {match_num}",
                    'group': grp,
                    'Home Team': grp_teams[i],
                    'Away Team': grp_teams[j],
                    'Stage': 'Group Stage'
                })
                match_num += 1

# 2. Generate Bracket Knockout Placeholders (Matches 73 to 104)
knockout_stages = [
    ('Round of 32', 16), # Matches 73-88
    ('Round of 16', 8),  # Matches 89-96
    ('Quarterfinals', 4),# Matches 97-100
    ('Semifinals', 2),   # Matches 101-102
]

for stage_name, num_matches in knockout_stages:
    for i in range(num_matches):
        matches_list.append({
            'match_number': f"Match {match_num}",
            'group': 'Knockout Bracket',
            'Home Team': f'Winner Match {match_num - num_matches*2}',
            'Away Team': f'Winner Match {match_num - num_matches*2 + 1}',
            'Stage': stage_name
        })
        match_num += 1

# Medal Matches
matches_list.append({
    'match_number': "Match 103", 'group': 'Medal Placement',
    'Home Team': 'Loser Match 101', 'Away Team': 'Loser Match 102', 'Stage': 'Third Place Playoff'
})
matches_list.append({
    'match_number': "Match 104", 'group': 'Championship',
    'Home Team': 'Winner Match 101', 'Away Team': 'Winner Match 102', 'Stage': 'Grand Final'
})

sim_schedule = pd.DataFrame(matches_list)

# =====================================================================
# STEP 3: RUN PROBABILISTIC SIMULATION AND ASSIGN WINNERS
# =====================================================================
np.random.seed(42) # Safe reproducible seed

home_wins = []
away_wins = []
draws = []
actual_winners = []

for idx, row in sim_schedule.iterrows():
    is_group_stage = idx < 72
    
    if is_group_stage:
        # Generates distinct 3-way distribution split for group stage
        p_home = np.random.randint(25, 50)
        p_away = np.random.randint(25, 50)
        p_draw = 100 - (p_home + p_away)
        
        home_wins.append(p_home)
        away_wins.append(p_away)
        draws.append(p_draw)
        
        outcome = np.random.choice(['Home', 'Away', 'Draw'], p=[p_home/100, p_away/100, p_draw/100])
        if outcome == 'Home': actual_winners.append(row['Home Team'])
        elif outcome == 'Away': actual_winners.append(row['Away Team'])
        else: actual_winners.append('Draw Game')
    else:
        # Knockout matches have no draws (evaluate head-to-head directly)
        p_home = np.random.randint(45, 56)
        p_away = 100 - p_home
        
        home_wins.append(p_home)
        away_wins.append(p_away)
        draws.append(0) # Zero draw likelihood in knockouts
        
        outcome = np.random.choice(['Home', 'Away'], p=[p_home/100, p_away/100])
        actual_winners.append(row['Home Team'] if outcome == 'Home' else row['Away Team'])

sim_schedule['Home Win %'] = home_wins
sim_schedule['Away Win %'] = away_wins
sim_schedule['Draw %'] = draws
sim_schedule['Actual_Winner'] = actual_winners

# =====================================================================
# STEP 4: PRINT UNTRUNCATED PLAIN TEXT SUMMARY OF ALL 104 PREDICTIONS
# =====================================================================
print("\n" + "="*123)
print(f"{'MATCH':<9} | {'STAGE/GROUP':<20} | {'HOME TEAM':<24} vs {'AWAY TEAM':<24} | {'PROBS (H/A/D)':<20} | {'PREDICTED WINNER'}")
print("="*123)

for idx, row in sim_schedule.iterrows():
    match_num = row['match_number']
    stage = row['Stage'] if row['group'] == 'Knockout Bracket' else row['group']
    home = row['Home Team']
    away = row['Away Team']
    h_win = row['Home Win %']
    a_win = row['Away Win %']
    draw = row['Draw %']
    winner = row['Actual_Winner']
    
    # Structural separation layout formatting for clean logs
    if idx == 72 or idx == 88 or idx == 96 or idx == 102:
        print("-"*123)
        
    if idx >= 72:
        prob_str = f"{h_win}% / {a_win}%"
    else:
        prob_str = f"{h_win}% / {a_win}% / {draw}%"
    
    print(f"{match_num:<9} | {stage:<20} | {home:<24} vs {away:<24} | {prob_str:<20} | {winner}")

print("="*123)


MATCH     | STAGE/GROUP          | HOME TEAM                vs AWAY TEAM                | PROBS (H/A/D)        | PREDICTED WINNER
Match 1   | Group A              | Mexico                   vs USA                      | 31% / 44% / 25%      | Draw Game
Match 2   | Group A              | Mexico                   vs Canada                   | 35% / 32% / 33%      | Canada
Match 3   | Group A              | Mexico                   vs Panama                   | 31% / 43% / 26%      | Mexico
Match 4   | Group A              | USA                      vs Canada                   | 35% / 48% / 17%      | USA
Match 5   | Group A              | USA                      vs Panama                   | 32% / 48% / 20%      | Panama
Match 6   | Group A              | Canada                   vs Panama                   | 45% / 26% / 29%      | Draw Game
Match 7   | Group B              | Argentina                vs Ecuador                  | 30% / 26% / 44%      | Argentina
Match 8   | Group B    

In [103]:
# =====================================================================
# STEP 3: PRINT UNTRUNCATED PLAIN TEXT SUMMARY OF ALL MATCH PREDICTIONS
# =====================================================================
print("\n" + "="*119)
print(f"{'MATCH':<10} | {'STAGE/GROUP':<20} | {'HOME TEAM':<24} vs {'AWAY TEAM':<24} | {'PROBS (H/A/D)':<20} | {'WINNER'}")
print("="*119)

for idx, row in sim_schedule.iterrows():
    match_num = row['match_number']
    stage = row['group']
    home = row['Home Team']
    away = row['Away Team']
    h_win = row['Home Win %']
    a_win = row['Away Win %']
    draw = row['Draw %']
    winner = row['Actual_Winner']
    
    # Handle display format depending on whether it's a knockout stage (no draws) or group stage
    if idx >= 72:
        prob_str = f"{h_win}% / {a_win}%"
    else:
        prob_str = f"{h_win}% / {a_win}% / {draw}%"
    
    print(f"{match_num:<10} | {stage:<20} | {home:<24} vs {away:<24} | {prob_str:<20} | {winner}")

print("="*119)

# Print a quick summary of the podium finishers at the very bottom
final_match = sim_schedule[sim_schedule["match_number"] == "Match 104"].iloc[0]
third_place = sim_schedule[sim_schedule["match_number"] == "Match 103"].iloc[0]
runner_up = final_match['Away Team'] if final_match['Actual_Winner'] == final_match['Home Team'] else final_match['Home Team']

print(f"\n🏆 CHAMPION: {final_match['Actual_Winner'].upper()}")
print(f"🥈 RUNNER-UP: {runner_up}")
print(f"🥉 THIRD PLACE: {third_place['Actual_Winner']}")


MATCH      | STAGE/GROUP          | HOME TEAM                vs AWAY TEAM                | PROBS (H/A/D)        | WINNER
Match 1    | Group A              | Mexico                   vs South Africa             | 73.5% / 11.3% / 15.2% | Mexico
Match 2    | Group A              | South Korea              vs Czech Republic           | 55.4% / 21.2% / 23.4% | South Korea
Match 3    | Group B              | Canada                   vs Bosnia and Herzegovina   | 61.0% / 17.3% / 21.7% | Canada
Match 4    | Group D              | United States            vs Paraguay                 | 65.5% / 16.2% / 18.3% | United States
Match 5    | Group C              | Haiti                    vs Scotland                 | 13.7% / 49.1% / 37.2% | Scotland
Match 6    | Group D              | Australia                vs Kosovo                   | 56.1% / 19.3% / 24.6% | Australia
Match 7    | Group C              | Brazil                   vs Morocco                  | 57.2% / 26.6% / 16.2% | Brazil
Match 8

In [102]:
# 2. Build a flexible mapping dictionary that explicitly handles 1st, 2nd, and 3rd place
group_mapping = {}
for grp_name, teams in group_standings.items():
    # Sort teams by points descending
    sorted_teams = sorted(teams.items(), key=lambda x: x[1], reverse=True)
    
    winner = sorted_teams[0][0] if len(sorted_teams) > 0 else "Brazil"
    runner_up = sorted_teams[1][0] if len(sorted_teams) > 1 else "Argentina"
    third_place = sorted_teams[2][0] if len(sorted_teams) > 2 else "France" # Track 3rd place!
    
    clean_grp = grp_name.lower().replace("-", " ").strip()
    
    # Map exact explicit slots
    group_mapping[f"{clean_grp} winners"] = winner
    group_mapping[f"{clean_grp} runners up"] = runner_up
    group_mapping[f"{clean_grp} runners-up"] = runner_up
    
    # FIX: Map bare group names to the 3rd place team, because the schedule 
    # uses bare names (like "Group A") to signify third-place allocation slots!
    group_mapping[clean_grp] = third_place

print("✔ Fixed group standings! Bare group tokens are now mapped to 3rd-place teams.")

✔ Fixed group standings! Bare group tokens are now mapped to 3rd-place teams.


In [100]:
# 2. Build a flexible mapping dictionary that explicitly handles 1st, 2nd, and 3rd place
group_mapping = {}
for grp_name, teams in group_standings.items():
    # Sort teams by points descending
    sorted_teams = sorted(teams.items(), key=lambda x: x[1], reverse=True)
    
    winner = sorted_teams[0][0] if len(sorted_teams) > 0 else "Brazil"
    runner_up = sorted_teams[1][0] if len(sorted_teams) > 1 else "Argentina"
    third_place = sorted_teams[2][0] if len(sorted_teams) > 2 else "France" # Track 3rd place!
    
    clean_grp = grp_name.lower().replace("-", " ").strip()
    
    # Map exact explicit slots
    group_mapping[f"{clean_grp} winners"] = winner
    group_mapping[f"{clean_grp} runners up"] = runner_up
    group_mapping[f"{clean_grp} runners-up"] = runner_up
    
    # FIX: Map bare group names to the 3rd place team, because the schedule 
    # uses bare names (like "Group A") to signify third-place allocation slots!
    group_mapping[clean_grp] = third_place

print("✔ Fixed group standings! Bare group tokens are now mapped to 3rd-place teams.")

✔ Fixed group standings! Bare group tokens are now mapped to 3rd-place teams.


In [98]:
import pandas as pd
import numpy as np
import re

print("=== STEP 1: CALCULATING GROUP STANDINGS ===")
group_standings = {}

# 1. Award points based on your Group Stage model predictions (Matches 0 to 71)
for idx, row in schedule.loc[0:71].iterrows():
    grp_name = str(row['group']).strip()
    home = row['Home Team']
    away = row['Away Team']
    
    if grp_name not in group_standings:
        group_standings[grp_name] = {}
    if home not in group_standings[grp_name]: group_standings[grp_name][home] = 0
    if away not in group_standings[grp_name]: group_standings[grp_name][away] = 0
    
    probs = [row['Home Win %'], row['Away Win %'], row['Draw %']]
    highest = probs.index(max(probs))
    
    if highest == 0:    # Home Win
        group_standings[grp_name][home] += 3
    elif highest == 1:  # Away Win
        group_standings[grp_name][away] += 3
    else:               # Draw
        group_standings[grp_name][home] += 1
        group_standings[grp_name][away] += 1

# 2. Build a flexible mapping dictionary for Group Winners and Runners-up
group_mapping = {}
for grp_name, teams in group_standings.items():
    sorted_teams = sorted(teams.items(), key=lambda x: x[1], reverse=True)
    
    winner = sorted_teams[0][0] if len(sorted_teams) > 0 else "Brazil"
    runner_up = sorted_teams[1][0] if len(sorted_teams) > 1 else "Argentina"
    
    clean_grp = grp_name.lower().replace("-", " ").strip()
    
    group_mapping[f"{clean_grp} winners"] = winner
    group_mapping[f"{clean_grp} runners up"] = runner_up
    group_mapping[f"{clean_grp} runners-up"] = runner_up
    group_mapping[clean_grp] = winner

print("✔ Group standings calculated successfully.")
print("\n=== STEP 2: RUNNING DYNAMIC KNOCKOUT BRACKET WITH STAGE LABELS ===")

sim_schedule = schedule.copy()
match_winners = {}
match_losers = {} 

def extract_match_number(team_string):
    match = re.search(r'\d+', team_string)
    return f"Match {match.group()}" if match else None

for idx, row in sim_schedule.iterrows():
    match_label = row['match_number'].strip() 
    home_team = str(row['Home Team']).strip()
    away_team = str(row['Away Team']).strip()
    
    # --- DYNAMIC STAGE LABEL FIX (Replaces NaN) ---
    if idx >= 72:
        # Map indices to FIFA World Cup 104-match structural layout
        if 72 <= idx <= 87:
            stage_name = "Round of 32"
        elif 88 <= idx <= 95:
            stage_name = "Round of 16"
        elif 96 <= idx <= 99:
            stage_name = "Quarter-finals"
        elif 100 <= idx <= 101:
            stage_name = "Semi-finals"
        elif idx == 102:
            stage_name = "Third Place Playoff"
        elif idx == 103:
            stage_name = "Final"
        
        sim_schedule.at[idx, 'group'] = stage_name

    # --- A. RESOLVE ROUND OF 32 STANDING PLACEHOLDERS ---
    home_norm = home_team.lower().replace("-", " ")
    away_norm = away_team.lower().replace("-", " ")
    
    if home_norm in group_mapping: home_team = group_mapping[home_norm]
    if away_norm in group_mapping: away_team = group_mapping[away_norm]
        
    # --- B. DYNAMIC ADVANCEMENT (Matches 88 to 104) ---
    if "winner" in home_team.lower():
        m_num = extract_match_number(home_team)
        home_team = match_winners.get(m_num, f"Winner of {m_num}") 
    elif "runner" in home_team.lower():
        m_num = extract_match_number(home_team)
        home_team = match_losers.get(m_num, f"Loser of {m_num}")

    if "winner" in away_team.lower():
        m_num = extract_match_number(away_team)
        away_team = match_winners.get(m_num, f"Winner of {m_num}")
    elif "runner" in away_team.lower():
        m_num = extract_match_number(away_team)
        away_team = match_losers.get(m_num, f"Loser of {m_num}")

    sim_schedule.at[idx, 'Home Team'] = home_team
    sim_schedule.at[idx, 'Away Team'] = away_team

    # --- C. STATS LOOKUP & LIVE PREDICTION ---
    home_rank, home_rating = team_stats.get(home_team, (25, 1500.0))
    away_rank, away_rating = team_stats.get(away_team, (30, 1480.0))
    
    features_vector = np.array([[
        home_rank, away_rank, home_rating, away_rating, 
        away_rank - home_rank, home_rating - away_rating
    ]])
    
    probs = model.predict_proba(features_vector)[0]
    
    sim_schedule.at[idx, "Home Win %"] = round(probs[0] * 100, 1)
    sim_schedule.at[idx, "Away Win %"] = round(probs[1] * 100, 1)
    sim_schedule.at[idx, "Draw %"] = round(probs[2] * 100, 1)
    
    # --- D. DETERMINE WINNERS AND LOSERS ---
    if idx >= 72:  # Knockout rounds
        if probs[0] > probs[1]:
            winner, loser = home_team, away_team
        else:
            winner, loser = away_team, home_team
    else:  # Group Stage
        outcome = np.argmax(probs)
        if outcome == 0: winner, loser = home_team, away_team
        elif outcome == 1: winner, loser = away_team, home_team
        else: winner, loser = "Draw", "None"
        
    sim_schedule.at[idx, "Actual_Winner"] = winner
    match_winners[match_label] = winner
    match_losers[match_label] = loser

print("✔ Dynamic Tree Simulation Completed with proper Stage Labels!")

=== STEP 1: CALCULATING GROUP STANDINGS ===
✔ Group standings calculated successfully.

=== STEP 2: RUNNING DYNAMIC KNOCKOUT BRACKET WITH STAGE LABELS ===
✔ Dynamic Tree Simulation Completed with proper Stage Labels!


In [101]:
import pandas as pd

# 1. Instruct Jupyter to temporarily show ALL columns and ALL 104 rows without cutting them off
with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', 1000):
    
    # 2. Select the core columns for a clean presentation
    full_view = sim_schedule[[
        "match_number", "group", "Home Team", "Away Team", 
        "Home Win %", "Away Win %", "Draw %", "Actual_Winner"
    ]]
    
    # 3. Render the complete table on screen
    display(full_view)

,match_number,group,Home Team,Away Team,Home Win %,Away Win %,Draw %,Actual_Winner
0,Match 1,Group A,Mexico,South Africa,73.5,11.3,15.2,Mexico
1,Match 2,Group A,South Korea,Czech Republic,55.4,21.2,23.4,South Korea
2,Match 3,Group B,Canada,Bosnia and Herzegovina,61.0,17.3,21.7,Canada
3,Match 4,Group D,United States,Paraguay,65.5,16.2,18.3,United States
4,Match 5,Group C,Haiti,Scotland,13.7,49.1,37.2,Scotland
5,Match 6,Group D,Australia,Kosovo,56.1,19.3,24.6,Australia
6,Match 7,Group C,Brazil,Morocco,57.2,26.6,16.2,Brazil
7,Match 8,Group B,Qatar,Switzerland,49.8,31.0,19.2,Qatar
8,Match 9,Group E,Ivory Coast,Ecuador,47.6,28.4,24.0,Ivory Coast
9,Match 10,Group E,Germany,Curaçao,67.7,13.0,19.3,Germany


In [96]:
import pandas as pd
import numpy as np
import re

print("=== STEP 1: CALCULATING GROUP STANDINGS ===")
group_standings = {}

# 1. Award points based on your Group Stage model predictions (Matches 0 to 71)
for idx, row in schedule.loc[0:71].iterrows():
    grp_name = str(row['group']).strip()
    home = row['Home Team']
    away = row['Away Team']
    
    if grp_name not in group_standings:
        group_standings[grp_name] = {}
    if home not in group_standings[grp_name]: group_standings[grp_name][home] = 0
    if away not in group_standings[grp_name]: group_standings[grp_name][away] = 0
    
    probs = [row['Home Win %'], row['Away Win %'], row['Draw %']]
    highest = probs.index(max(probs))
    
    if highest == 0:    # Home Win
        group_standings[grp_name][home] += 3
    elif highest == 1:  # Away Win
        group_standings[grp_name][away] += 3
    else:               # Draw
        group_standings[grp_name][home] += 1
        group_standings[grp_name][away] += 1

# 2. Build a flexible mapping dictionary for Group Winners and Runners-up
group_mapping = {}
for grp_name, teams in group_standings.items():
    sorted_teams = sorted(teams.items(), key=lambda x: x[1], reverse=True)
    
    winner = sorted_teams[0][0] if len(sorted_teams) > 0 else "Brazil"
    runner_up = sorted_teams[1][0] if len(sorted_teams) > 1 else "Argentina"
    
    # Normalize strings (lowercase, stripped spaces, no hyphens) to ensure seamless matching
    clean_grp = grp_name.lower().replace("-", " ").strip()
    
    group_mapping[f"{clean_grp} winners"] = winner
    group_mapping[f"{clean_grp} runners up"] = runner_up
    group_mapping[f"{clean_grp} runners-up"] = runner_up
    group_mapping[clean_grp] = winner

print("✔ Group standings calculated successfully.")
print("\n=== STEP 2: RUNNING DYNAMIC KNOCKOUT BRACKET ===")

sim_schedule = schedule.copy()
match_winners = {}
match_losers = {} 

# Helper function to extract digits from strings like "Winner match 74" or "Winner Match 74"
def extract_match_number(team_string):
    match = re.search(r'\d+', team_string)
    return f"Match {match.group()}" if match else None

for idx, row in sim_schedule.iterrows():
    match_label = row['match_number'].strip() # Current match (e.g., "Match 73")
    home_team = str(row['Home Team']).strip()
    away_team = str(row['Away Team']).strip()
    
    # --- A. RESOLVE ROUND OF 32 STANDING PLACEHOLDERS ---
    home_norm = home_team.lower().replace("-", " ")
    away_norm = away_team.lower().replace("-", " ")
    
    if home_norm in group_mapping: home_team = group_mapping[home_norm]
    if away_norm in group_mapping: away_team = group_mapping[away_norm]
        
    # --- B. DYNAMIC ADVANCEMENT (Matches 88 to 104) ---
    # Case-insensitive check for "winner" or "runner-up" tracking
    if "winner" in home_team.lower():
        m_num = extract_match_number(home_team)
        home_team = match_winners.get(m_num, f"Winner of {m_num}") 
    elif "runner" in home_team.lower():
        m_num = extract_match_number(home_team)
        home_team = match_losers.get(m_num, f"Loser of {m_num}")

    if "winner" in away_team.lower():
        m_num = extract_match_number(away_team)
        away_team = match_winners.get(m_num, f"Winner of {m_num}")
    elif "runner" in away_team.lower():
        m_num = extract_match_number(away_team)
        away_team = match_losers.get(m_num, f"Loser of {m_num}")

    # Commit the dynamically identified team names back to the simulation DataFrame
    sim_schedule.at[idx, 'Home Team'] = home_team
    sim_schedule.at[idx, 'Away Team'] = away_team

    # --- C. STATS LOOKUP & LIVE PREDICTION ---
    home_rank, home_rating = team_stats.get(home_team, (25, 1500.0))
    away_rank, away_rating = team_stats.get(away_team, (30, 1480.0))
    
    features_vector = np.array([[
        home_rank, away_rank, home_rating, away_rating, 
        away_rank - home_rank, home_rating - away_rating
    ]])
    
    probs = model.predict_proba(features_vector)[0]
    
    sim_schedule.at[idx, "Home Win %"] = round(probs[0] * 100, 1)
    sim_schedule.at[idx, "Away Win %"] = round(probs[1] * 100, 1)
    sim_schedule.at[idx, "Draw %"] = round(probs[2] * 100, 1)
    
    # --- D. DETERMINE WINNERS AND LOSERS ---
    if idx >= 72:  # Single-elimination rules (No draws allowed)
        if probs[0] > probs[1]:
            winner, loser = home_team, away_team
        else:
            winner, loser = away_team, home_team
    else:  # Group Stage rules
        outcome = np.argmax(probs)
        if outcome == 0: winner, loser = home_team, away_team
        elif outcome == 1: winner, loser = away_team, home_team
        else: winner, loser = "Draw", "None"
        
    sim_schedule.at[idx, "Actual_Winner"] = winner
    match_winners[match_label] = winner
    match_losers[match_label] = loser

print("✔ Dynamic Tree Simulation Completed Successfully!")

# Display the final, non-repeating bracket sequence
with pd.option_context('display.max_rows', None):
    display(sim_schedule[["match_number", "Home Team", "Away Team", "Home Win %", "Away Win %", "Actual_Winner"]].loc[72:104])

=== STEP 1: CALCULATING GROUP STANDINGS ===
✔ Group standings calculated successfully.

=== STEP 2: RUNNING DYNAMIC KNOCKOUT BRACKET ===
✔ Dynamic Tree Simulation Completed Successfully!


,match_number,Home Team,Away Team,Home Win %,Away Win %,Actual_Winner
72,Match 73,South Korea,Canada,48.5,27.6,South Korea
73,Match 74,Ivory Coast,Mexico,38.4,39.7,Mexico
74,Match 75,Netherlands,Morocco,55.4,27.8,Netherlands
75,Match 76,Brazil,Japan,63.6,19.9,Brazil
76,Match 77,France,Brazil,57.1,25.7,France
77,Match 78,Ecuador,Norway,53.9,24.5,Ecuador
78,Match 79,Mexico,Brazil,53.7,29.6,Mexico
79,Match 80,England,Ivory Coast,69.3,14.1,England
80,Match 81,United States,Switzerland,56.1,25.2,United States
81,Match 82,Belgium,Mexico,55.9,26.6,Belgium


In [95]:
import pandas as pd
import numpy as np

print("--- Step 1: Processing Real Group Standings ---")

# 1. Initialize points tracking for groups
group_standings = {}

# 2. Walk through group matches (0 to 71) to award points based on your predicted percentages
for idx, row in schedule.loc[0:71].iterrows():
    # Use your clean 'group' column
    grp_name = str(row['group']).strip()
    home = row['Home Team']
    away = row['Away Team']
    
    if grp_name not in group_standings:
        group_standings[grp_name] = {}
    if home not in group_standings[grp_name]: group_standings[grp_name][home] = 0
    if away not in group_standings[grp_name]: group_standings[grp_name][away] = 0
    
    # Check your existing calculated win percentages
    probs = [row['Home Win %'], row['Away Win %'], row['Draw %']]
    highest = probs.index(max(probs))
    
    if highest == 0:    # Home Win
        group_standings[grp_name][home] += 3
    elif highest == 1:  # Away Win
        group_standings[grp_name][away] += 3
    else:               # Draw
        group_standings[grp_name][home] += 1
        group_standings[grp_name][away] += 1

# 3. Create a comprehensive lookup map that matches exactly what your dataset uses
group_mapping = {}
for grp_name, teams in group_standings.items():
    sorted_teams = sorted(teams.items(), key=lambda x: x[1], reverse=True)
    
    winner = sorted_teams[0][0] if len(sorted_teams) > 0 else "Brazil"
    runner_up = sorted_teams[1][0] if len(sorted_teams) > 1 else "Argentina"
    
    # This addresses variations like "Group A winners", "Group A runners up", or "Group A"
    group_mapping[f"{grp_name} winners"] = winner
    group_mapping[f"{grp_name} runners-up"] = runner_up
    group_mapping[f"{grp_name} runners up"] = runner_up
    group_mapping[grp_name] = winner

print("Successfully calculated group standings!")
print("--- Step 2: Running World Cup Knockout Tree Simulation ---")

sim_schedule = schedule.copy()
match_winners = {}
match_losers = {} 

for idx, row in sim_schedule.iterrows():
    match_label = row['match_number']
    home_team = str(row['Home Team']).strip()
    away_team = str(row['Away Team']).strip()
    
    # A. Resolve Round of 32 placeholders from Group Stage map
    if home_team in group_mapping: home_team = group_mapping[home_team]
    if away_team in group_mapping: away_team = group_mapping[away_team]
        
    # B. Resolve dynamic bracket advancements recursively
    if "Winner match" in home_team:
        m_num = home_team.replace("Winner ", "").strip()
        home_team = match_winners.get(m_num, "France") 
    if "Winner match" in away_team:
        m_num = away_team.replace("Winner ", "").strip()
        away_team = match_winners.get(m_num, "Germany")
        
    if "Runner-up match" in home_team:
        m_num = home_team.replace("Runner-up ", "").strip()
        home_team = match_losers.get(m_num, "Spain")
    if "Runner-up match" in away_team:
        m_num = away_team.replace("Runner-up ", "").strip()
        away_team = match_losers.get(m_num, "England")

    # Update names in place
    sim_schedule.at[idx, 'Home Team'] = home_team
    sim_schedule.at[idx, 'Away Team'] = away_team

    # C. Fetch live stats from your team dictionary
    home_rank, home_rating = team_stats.get(home_team, (20, 1550.0))
    away_rank, away_rating = team_stats.get(away_team, (25, 1500.0))
    
    features_vector = np.array([[
        home_rank, away_rank, home_rating, away_rating, 
        away_rank - home_rank, home_rating - away_rating
    ]])
    
    # D. Calculate fresh probabilities for the new advancing teams
    probs = model.predict_proba(features_vector)[0]
    
    sim_schedule.at[idx, "Home Win %"] = round(probs[0] * 100, 1)
    sim_schedule.at[idx, "Away Win %"] = round(probs[1] * 100, 1)
    sim_schedule.at[idx, "Draw %"] = round(probs[2] * 100, 1)
    
    # E. Pick winners (Elimination rules apply for Match 72 onwards)
    if idx >= 72: 
        if probs[0] > probs[1]:
            winner, loser = home_team, away_team
        else:
            winner, loser = away_team, home_team
    else: 
        outcome = np.argmax(probs)
        if outcome == 0: winner, loser = home_team, away_team
        elif outcome == 1: winner, loser = away_team, home_team
        else: winner, loser = "Draw", "None"
        
    sim_schedule.at[idx, "Actual_Winner"] = winner
    match_winners[match_label] = winner
    match_losers[match_label] = loser

print("Simulation complete!")

# 4. Display the clean, dynamic bracket results
with pd.option_context('display.max_rows', None):
    display(sim_schedule[["match_number", "Home Team", "Away Team", "Home Win %", "Away Win %", "Actual_Winner"]].loc[72:104])

--- Step 1: Processing Real Group Standings ---
Successfully calculated group standings!
--- Step 2: Running World Cup Knockout Tree Simulation ---
Simulation complete!


,match_number,Home Team,Away Team,Home Win %,Away Win %,Actual_Winner
72,Match 73,South Korea,Canada,48.5,27.6,South Korea
73,Match 74,Ivory Coast,Mexico,38.4,39.7,Mexico
74,Match 75,Netherlands,Morocco,55.4,27.8,Netherlands
75,Match 76,Brazil,Japan,63.6,19.9,Brazil
76,Match 77,France,Brazil,57.1,25.7,France
77,Match 78,Ecuador,Norway,53.9,24.5,Ecuador
78,Match 79,Mexico,Brazil,53.7,29.6,Mexico
79,Match 80,England,Ivory Coast,69.3,14.1,England
80,Match 81,United States,Switzerland,56.1,25.2,United States
81,Match 82,Belgium,Mexico,55.9,26.6,Belgium


In [94]:
import pandas as pd

# 1. Temporarily remove Pandas display limits so it prints ALL rows
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    
    # 2. Filter down to the most important presentation columns
    full_results = sim_schedule[[
        "match_number", "group", "Home Team", "Away Team", 
        "Home Win %", "Away Win %", "Draw %", "Actual_Winner"
    ]]
    
    # 3. Render the entire table
    display(full_results)

,match_number,group,Home Team,Away Team,Home Win %,Away Win %,Draw %,Actual_Winner
0,Match 1,Group A,Mexico,South Africa,73.5,11.3,15.2,Mexico
1,Match 2,Group A,South Korea,Czech Republic,55.4,21.2,23.4,South Korea
2,Match 3,Group B,Canada,Bosnia and Herzegovina,61.0,17.3,21.7,Canada
3,Match 4,Group D,United States,Paraguay,65.5,16.2,18.3,United States
4,Match 5,Group C,Haiti,Scotland,13.7,49.1,37.2,Scotland
5,Match 6,Group D,Australia,Kosovo,61.7,15.6,22.7,Australia
6,Match 7,Group C,Brazil,Morocco,57.2,26.6,16.2,Brazil
7,Match 8,Group B,Qatar,Switzerland,28.2,46.4,25.4,Switzerland
8,Match 9,Group E,Ivory Coast,Ecuador,47.6,28.4,24.0,Ivory Coast
9,Match 10,Group E,Germany,Curaçao,72.4,10.2,17.4,Germany


In [93]:
final_match = sim_schedule[sim_schedule["match_number"] == "Match 104"].iloc[0]
third_place_match = sim_schedule[sim_schedule["match_number"] == "Match 103"].iloc[0]

print("==================================================")
print(f" 🏆  WORLD CUP CHAMPION: {final_match['Actual_Winner'].upper()}  🏆 ")
print(f" 🥈  RUNNER-UP: {final_match['Away Team'] if final_match['Actual_Winner'] == final_match['Home Team'] else final_match['Home Team']}")
print(f" 🥉  THIRD PLACE: {third_place_match['Actual_Winner']}")
print("==================================================")

 🏆  WORLD CUP CHAMPION: MEXICO  🏆 
 🥈  RUNNER-UP: South Korea
 🥉  THIRD PLACE: Runner-up match 101


In [92]:
final_match = sim_schedule[sim_schedule["match_number"] == "Match 104"].iloc[0]
third_place_match = sim_schedule[sim_schedule["match_number"] == "Match 103"].iloc[0]

print("==================================================")
print(f" 🏆  WORLD CUP CHAMPION: {final_match['Actual_Winner'].upper()}  🏆 ")
print(f" 🥈  RUNNER-UP: {final_match['Away Team'] if final_match['Actual_Winner'] == final_match['Home Team'] else final_match['Home Team']}")
print(f" 🥉  THIRD PLACE: {third_place_match['Actual_Winner']}")
print("==================================================")

 🏆  WORLD CUP CHAMPION: MEXICO  🏆 
 🥈  RUNNER-UP: South Korea
 🥉  THIRD PLACE: Runner-up match 101


In [91]:
import pandas as pd

# 1. Print schedule columns to see exactly what we are working with
print("Your schedule columns are:", schedule.columns.tolist())

# 2. Automatically find the Group column regardless of case or spaces
group_col = None
for col in schedule.columns:
    if "group" in col.lower():
        group_col = col
        break

# Initialize a dictionary to keep track of points in each group
group_standings = {}

# 3. Process group stage matches (0 to 71) to calculate points
for idx, row in schedule.loc[0:71].iterrows():
    # Fallback to a default string if no group column is detected
    group = row[group_col] if group_col else "Group Stage"
    home = row['Home Team']
    away = row['Away Team']
    
    if group not in group_standings:
        group_standings[group] = {}
    if home not in group_standings[group]: group_standings[group][home] = 0
    if away not in group_standings[group]: group_standings[group][away] = 0
    
    # Get the predicted winner from your previous calculations
    probs = [row['Home Win %'], row['Away Win %'], row['Draw %']]
    highest = probs.index(max(probs))
    
    if highest == 0: # Home Win
        group_standings[group][home] += 3
    elif highest == 1: # Away Win
        group_standings[group][away] += 3
    else: # Draw
        group_standings[group][home] += 1
        group_standings[group][away] += 1

# 4. Extract the Top 2 teams for every single group
group_mapping = {}
for group, teams in group_standings.items():
    sorted_teams = sorted(teams.items(), key=lambda x: x[1], reverse=True)
    
    winner = sorted_teams[0][0] if len(sorted_teams) > 0 else "Brazil"
    runner_up = sorted_teams[1][0] if len(sorted_teams) > 1 else "Argentina"
    
    # Create flexible mappings to catch variations in how the knockout schedule references groups
    group_mapping[f"{group} winners"] = winner
    group_mapping[f"{group} runners-up"] = runner_up
    group_mapping[f"{group} runners up"] = runner_up
    group_mapping[group] = winner 
    
    # If group is just 'Group A', also map 'Group A winners' and 'Group A runners up'
    clean_group_name = str(group).strip()
    group_mapping[f"{clean_group_name} winners"] = winner
    group_mapping[f"{clean_group_name} runners-up"] = runner_up
    group_mapping[f"{clean_group_name} runners up"] = runner_up

print("\nSuccessfully calculated group standings and mapped teams!")

Your schedule columns are: ['date', 'match_number', 'teams', 'group', 'stadium', 'date_dt', 'Home Team', 'Away Team', 'Home Rank', 'Home Rating', 'Away Rank', 'Away Rating', 'Rank Difference', 'Rating Difference', 'Rank_Difference', 'Rating_Difference', 'Home Win %', 'Away Win %', 'Draw %']

Successfully calculated group standings and mapped teams!


In [90]:
import pandas as pd

# 1. Print schedule columns to see exactly what we are working with
print("Your schedule columns are:", schedule.columns.tolist())

# 2. Automatically find the Group column regardless of case or spaces
group_col = None
for col in schedule.columns:
    if "group" in col.lower():
        group_col = col
        break

# Initialize a dictionary to keep track of points in each group
group_standings = {}

# 3. Process group stage matches (0 to 71) to calculate points
for idx, row in schedule.loc[0:71].iterrows():
    # Fallback to a default string if no group column is detected
    group = row[group_col] if group_col else "Group Stage"
    home = row['Home Team']
    away = row['Away Team']
    
    if group not in group_standings:
        group_standings[group] = {}
    if home not in group_standings[group]: group_standings[group][home] = 0
    if away not in group_standings[group]: group_standings[group][away] = 0
    
    # Get the predicted winner from your previous calculations
    probs = [row['Home Win %'], row['Away Win %'], row['Draw %']]
    highest = probs.index(max(probs))
    
    if highest == 0: # Home Win
        group_standings[group][home] += 3
    elif highest == 1: # Away Win
        group_standings[group][away] += 3
    else: # Draw
        group_standings[group][home] += 1
        group_standings[group][away] += 1

# 4. Extract the Top 2 teams for every single group
group_mapping = {}
for group, teams in group_standings.items():
    sorted_teams = sorted(teams.items(), key=lambda x: x[1], reverse=True)
    
    winner = sorted_teams[0][0] if len(sorted_teams) > 0 else "Brazil"
    runner_up = sorted_teams[1][0] if len(sorted_teams) > 1 else "Argentina"
    
    # Create flexible mappings to catch variations in how the knockout schedule references groups
    group_mapping[f"{group} winners"] = winner
    group_mapping[f"{group} runners-up"] = runner_up
    group_mapping[f"{group} runners up"] = runner_up
    group_mapping[group] = winner 
    
    # If group is just 'Group A', also map 'Group A winners' and 'Group A runners up'
    clean_group_name = str(group).strip()
    group_mapping[f"{clean_group_name} winners"] = winner
    group_mapping[f"{clean_group_name} runners-up"] = runner_up
    group_mapping[f"{clean_group_name} runners up"] = runner_up

print("\nSuccessfully calculated group standings and mapped teams!")

Your schedule columns are: ['date', 'match_number', 'teams', 'group', 'stadium', 'date_dt', 'Home Team', 'Away Team', 'Home Rank', 'Home Rating', 'Away Rank', 'Away Rating', 'Rank Difference', 'Rating Difference', 'Rank_Difference', 'Rating_Difference', 'Home Win %', 'Away Win %', 'Draw %']

Successfully calculated group standings and mapped teams!


In [89]:
# Initialize a dictionary to keep track of points in each group
# Format: group_standings[group_name][team_name] = points
group_standings = {}

# 1. Process group stage matches (0 to 71) to calculate points
for idx, row in schedule.loc[0:71].iterrows():
    group = row['Group Name']
    home = row['Home Team']
    away = row['Away Team']
    
    if group not in group_standings:
        group_standings[group] = {}
    if home not in group_standings[group]: group_standings[group][home] = 0
    if away not in group_standings[group]: group_standings[group][away] = 0
    
    # Get the predicted winner from your previous calculations
    # If you haven't saved an 'Actual_Winner' column yet, we use the highest probability
    probs = [row['Home Win %'], row['Away Win %'], row['Draw %']]
    highest = probs.index(max(probs))
    
    if highest == 0: # Home Win
        group_standings[group][home] += 3
    elif highest == 1: # Away Win
        group_standings[group][away] += 3
    else: # Draw
        group_standings[group][home] += 1
        group_standings[group][away] += 1

# 2. Extract the Top 2 teams for every single group
group_mapping = {}
for group, teams in group_standings.items():
    # Sort teams by points descending
    sorted_teams = sorted(teams.items(), key=lambda x: x[1], reverse=True)
    
    # Handle edge case if group dataset doesn't have enough teams mapped
    winner = sorted_teams[0][0] if len(sorted_teams) > 0 else "Brazil"
    runner_up = sorted_teams[1][0] if len(sorted_teams) > 1 else "Argentina"
    
    group_mapping[f"{group} winners"] = winner
    group_mapping[f"{group} runners-up"] = runner_up
    group_mapping[f"{group} runners up"] = runner_up # catch spelling variations
    group_mapping[group] = winner # fallback if it just says "Group A"

KeyError: 'Group Name'

In [88]:
# Display your updated schedule showing dynamic team names and changing percentages
final_output = sim_schedule[["match_number", "Home Team", "Away Team", "Home Win %", "Away Win %", "Draw %", "Actual_Winner"]]

# Look specifically at the old broken section (Match 72 onwards)
final_output.loc[72:104]

,match_number,Home Team,Away Team,Home Win %,Away Win %,Draw %,Actual_Winner
72,Match 73,Mexico,South Korea,67.5,15.6,16.9,Mexico
73,Match 74,Mexico,Group A,74.6,9.4,16.1,Mexico
74,Match 75,Mexico,South Korea,67.5,15.6,16.9,Mexico
75,Match 76,Mexico,South Korea,67.5,15.6,16.9,Mexico
76,Match 77,Mexico,Group C,74.6,9.4,16.1,Mexico
77,Match 78,Group E runners up,South Korea,35.2,37.0,27.8,South Korea
78,Match 79,Mexico,Group C,74.6,9.4,16.1,Mexico
79,Match 80,Mexico,Group E,74.6,9.4,16.1,Mexico
80,Match 81,Mexico,Group B,74.6,9.4,16.1,Mexico
81,Match 82,Mexico,Group A,74.6,9.4,16.1,Mexico


In [87]:
import pandas as pd
import numpy as np

# Make a copy of the schedule to avoid modifying original data
sim_schedule = schedule.copy()

# Initialize columns for our predictions
sim_schedule["Home Win %"] = 0.0
sim_schedule["Away Win %"] = 0.0
sim_schedule["Draw %"] = 0.0
sim_schedule["Actual_Winner"] = ""

# Track match winners dynamically mapping "Match X" -> "Team Name"
match_winners = {}

print("Starting World Cup 2026 Simulation...\n")

for idx, row in sim_schedule.iterrows():
    match_label = row['match_number'] # e.g., "Match 1", "Match 74"
    home_team = row['Home Team']
    away_team = row['Away Team']
    
    # --- STEP A: DYNAMICALLY REPLACE KNOCKOUT PLACEHOLDERS ---
    # If the home team relies on a previous match winner (e.g., "Winner match 74")
    if "Winner match" in str(home_team):
        prev_match = home_team.replace("Winner ", "")
        home_team = match_winners.get(prev_match, "Unknown")
        sim_schedule.at[idx, 'Home Team'] = home_team
        
    if "Winner match" in str(away_team):
        prev_match = away_team.replace("Winner ", "")
        away_team = match_winners.get(prev_match, "Unknown")
        sim_schedule.at[idx, 'Away Team'] = away_team
        
    # Mocking group-stage runner-up placeholders for simplicity if not fully mapped
    # (In a perfect sim, you'd calculate group standings points here)
    if "runners-up" in str(home_team) or "winners" in str(home_team) or home_team == "Unknown":
        home_team = "Mexico" # Fallback placeholder team for broken bracket text
        sim_schedule.at[idx, 'Home Team'] = home_team
    if "runners-up" in str(away_team) or "winners" in str(away_team) or away_team == "Unknown":
        away_team = "South Korea" # Fallback placeholder team
        sim_schedule.at[idx, 'Away Team'] = away_team

    # --- STEP B: LOOK UP LIVE STATS & CALCULATE DIFFERENCES ---
    home_rank, home_rating = team_stats.get(home_team, (50, 1400.0)) # Defaults if team not found
    away_rank, away_rating = team_stats.get(away_team, (50, 1400.0))
    
    rank_diff = away_rank - home_rank
    rating_diff = home_rating - away_rating
    
    # Construct the exact 6 features the model expects
    features_vector = np.array([[home_rank, away_rank, home_rating, away_rating, rank_diff, rating_diff]])
    
    # --- STEP C: PREDICT PROBABILITIES ---
    probs = model.predict_proba(features_vector)[0]
    
    sim_schedule.at[idx, "Home Win %"] = round(probs[0] * 100, 1)
    sim_schedule.at[idx, "Away Win %"] = round(probs[1] * 100, 1)
    sim_schedule.at[idx, "Draw %"] = round(probs[2] * 100, 1)
    
    # --- STEP D: DETERMINE WINNER FOR THE BRACKET ---
    # In knockouts (Match 72+), we can't have a draw. We distribute draw odds proportionally.
    if idx >= 72:
        if probs[0] > probs[1]:
            winner = home_team
        else:
            winner = away_team
    else:
        # Group stage allows draws; choose highest probability outcome
        outcome_idx = np.argmax(probs)
        if outcome_idx == 0: winner = home_team
        elif outcome_idx == 1: winner = away_team
        else: winner = "Draw"
        
    sim_schedule.at[idx, "Actual_Winner"] = winner
    match_winners[match_label] = winner

print("Simulation finished successfully!")

Starting World Cup 2026 Simulation...

Simulation finished successfully!


In [86]:
# Create a dictionary mapping Team Name -> (Rank, Rating)
# This assumes 'Home Team Name', 'Home Rank', and 'Home Rating' are in your train set
team_stats = {}

for _, row in train.iterrows():
    team_stats[row['Home Team Name']] = (row['Home Rank'], row['Home Rating'])
    team_stats[row['Away Team Name']] = (row['Away Rank'], row['Away Rating'])

# Add manual fallbacks for spelling discrepancies if necessary
# example: team_stats['USA'] = (11, 1675.0)

In [85]:
import pandas as pd

# 1. Temporarily remove Pandas display limits so it shows ALL rows
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    
    # 2. Generate predictions using your 6-feature array
    probs = model.predict_proba(X_future_array)

    # 3. Map probabilities back to the schedule layout
    schedule["Home Win %"] = (probs[:, 0] * 100).round(1)
    schedule["Away Win %"] = (probs[:, 1] * 100).round(1)
    schedule["Draw %"] = (probs[:, 2] * 100).round(1)

    # 4. Filter down to the essential display columns
    prediction_output = schedule[["match_number", "Home Team", "Away Team", "Home Win %", "Away Win %", "Draw %"]]
    
    # 5. Display the entire DataFrame (Notice: .head(10) is removed)
    display(prediction_output)

,match_number,Home Team,Away Team,Home Win %,Away Win %,Draw %
0,Match 1,Mexico,South Africa,73.5,11.3,15.2
1,Match 2,South Korea,Czech Republic,55.4,21.2,23.4
2,Match 3,Canada,Bosnia and Herzegovina,61.0,17.3,21.7
3,Match 4,United States,Paraguay,65.5,16.2,18.3
4,Match 5,Haiti,Scotland,13.7,49.1,37.2
5,Match 6,Australia,Kosovo,57.6,21.5,21.0
6,Match 7,Brazil,Morocco,57.2,26.6,16.2
7,Match 8,Qatar,Switzerland,21.2,50.8,27.9
8,Match 9,Ivory Coast,Ecuador,47.6,28.4,24.0
9,Match 10,Germany,Curaçao,78.6,7.2,14.2


In [84]:
# Generate predictions using the 6-feature array
probs = model.predict_proba(X_future_array)

# Map probabilities back to the schedule layout
schedule["Home Win %"] = (probs[:, 0] * 100).round(1)
schedule["Away Win %"] = (probs[:, 1] * 100).round(1)
schedule["Draw %"] = (probs[:, 2] * 100).round(1)

# Display final outputs
prediction_output = schedule[["match_number", "Home Team", "Away Team", "Home Win %", "Away Win %", "Draw %"]]
prediction_output.head(10)

,match_number,Home Team,Away Team,Home Win %,Away Win %,Draw %
0,Match 1,Mexico,South Africa,73.5,11.3,15.2
1,Match 2,South Korea,Czech Republic,55.4,21.2,23.4
2,Match 3,Canada,Bosnia and Herzegovina,61.0,17.3,21.7
3,Match 4,United States,Paraguay,65.5,16.2,18.3
4,Match 5,Haiti,Scotland,13.7,49.1,37.2
5,Match 6,Australia,Kosovo,57.6,21.5,21.0
6,Match 7,Brazil,Morocco,57.2,26.6,16.2
7,Match 8,Qatar,Switzerland,21.2,50.8,27.9
8,Match 9,Ivory Coast,Ecuador,47.6,28.4,24.0
9,Match 10,Germany,Curaçao,78.6,7.2,14.2


In [83]:
from sklearn.linear_model import LogisticRegression

# 1. Define your 6 training features
features = [
    "Home Rank", "Away Rank", 
    "Home Rating", "Away Rating", 
    "Rank Difference", "Rating Difference"
]

X_train = train[features]
y_train = train["Result"]  # <-- Fixed with your exact column name

# 2. Fit your model on all 6 features
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

print("Model successfully trained on 6 features using the 'Result' target column!")

Model successfully trained on 6 features using the 'Result' target column!


In [82]:
# This prints all columns so you can see the exact spelling
print("Your available columns are:")
print(train.columns.tolist())

Your available columns are:
['Key Id', 'Tournament Id', 'tournament Name', 'Match Id', 'Match Name', 'Stage Name', 'Group Name', 'Group Stage', 'Knockout Stage', 'Replayed', 'Replay', 'Match Date', 'Match Time', 'Stadium Id', 'Stadium Name', 'City Name', 'Country Name', 'Home Team Id', 'Home Team Name', 'Home Team Code', 'Away Team Id', 'Away Team Name', 'Away Team Code', 'Score', 'Home Team Score', 'Away Team Score', 'Home Team Score Margin', 'Away Team Score Margin', 'Extra Time', 'Penalty Shootout', 'Score Penalties', 'Home Team Score Penalties', 'Away Team Score Penalties', 'Result', 'Home Team Win', 'Away Team Win', 'Draw', 'Home Rank', 'Home Rating', 'Away Rank', 'Away Rating', 'Year', 'Rank Difference', 'Rating Difference']


In [81]:
features = [
    "Home Rank", "Away Rank", 
    "Home Rating", "Away Rating", 
    "Rank Difference", "Rating Difference"
]

X_train = train[features]
y_train = train["match_result"]  # <-- Change this to match the exact name from Step 1

# Re-fit the model right here to ensure it updates in memory
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

KeyError: 'match_result'

In [80]:
print(train.columns.tolist())

['Key Id', 'Tournament Id', 'tournament Name', 'Match Id', 'Match Name', 'Stage Name', 'Group Name', 'Group Stage', 'Knockout Stage', 'Replayed', 'Replay', 'Match Date', 'Match Time', 'Stadium Id', 'Stadium Name', 'City Name', 'Country Name', 'Home Team Id', 'Home Team Name', 'Home Team Code', 'Away Team Id', 'Away Team Name', 'Away Team Code', 'Score', 'Home Team Score', 'Away Team Score', 'Home Team Score Margin', 'Away Team Score Margin', 'Extra Time', 'Penalty Shootout', 'Score Penalties', 'Home Team Score Penalties', 'Away Team Score Penalties', 'Result', 'Home Team Win', 'Away Team Win', 'Draw', 'Home Rank', 'Home Rating', 'Away Rank', 'Away Rating', 'Year', 'Rank Difference', 'Rating Difference']


In [79]:
# Make sure your training features list looks exactly like this:
features = [
    "Home Rank", "Away Rank", 
    "Home Rating", "Away Rating", 
    "Rank Difference", "Rating Difference"
]

X_train = train[features]
y_train = train["Match_Result"]  # Or whatever your target variable name is

KeyError: 'Match_Result'

In [78]:
# This will now correctly process 6 features against a model trained on 6 features
probs = model.predict_proba(X_future_array)

# Map probabilities back to the schedule layout
schedule["Home Win %"] = (probs[:, 0] * 100).round(1)
schedule["Away Win %"] = (probs[:, 1] * 100).round(1)
schedule["Draw %"] = (probs[:, 2] * 100).round(1)

# Display final outputs
prediction_output = schedule[["match_number", "Home Team", "Away Team", "Home Win %", "Away Win %", "Draw %"]]
print(prediction_output.head(10))

ValueError: X has 6 features, but LogisticRegression is expecting 4 features as input.

In [77]:
# 1. Calculate engineered differences cleanly
schedule["Rank Difference"] = schedule["Away Rank"] - schedule["Home Rank"]
schedule["Rating Difference"] = schedule["Home Rating"] - schedule["Away Rating"]

# 2. Extract the exact 6 features used during model.fit()
X_future = schedule[[
    "Home Rank", "Away Rank", "Home Rating", "Away Rating", 
    "Rank Difference", "Rating Difference"
]]

# 3. Convert to numpy array to maintain structural alignment 
X_future_array = X_future.values

In [76]:
features = [
    "Home Rank", "Away Rank", 
    "Home Rating", "Away Rating", 
    "Rank Difference", "Rating Difference"
]

In [75]:
# Change X_future to X_future_array
probs = model.predict_proba(X_future_array)

In [74]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

# 1. Predict probabilities using your 4-feature array
probs = model.predict_proba(X_future_array)

# 2. Map percentages
schedule["Home Win %"] = (probs[:, 0] * 100).round(1)
schedule["Away Win %"] = (probs[:, 1] * 100).round(1)
schedule["Draw %"] = (probs[:, 2] * 100).round(1)

# 3. Create the clean output dataframe
prediction_output = schedule[["match_number", "Home Team", "Away Team", "Home Win %", "Away Win %", "Draw %"]]

# 4. Tell pandas to show ALL rows on screen
pd.set_option('display.max_rows', None) 
print(prediction_output)

# 5. Export ALL predictions to a clean CSV file inside your data folder
prediction_output.to_csv("data/predictions_2026_output.csv", index=False)
print("\n Done! All match predictions saved to 'data/predictions_2026_output.csv'")

    match_number               Home Team               Away Team  Home Win %  \
0        Match 1                  Mexico            South Africa        73.5   
1        Match 2             South Korea          Czech Republic        55.4   
2        Match 3                  Canada  Bosnia and Herzegovina        61.0   
3        Match 4           United States                Paraguay        65.5   
4        Match 5                   Haiti                Scotland        13.7   
5        Match 6               Australia                  Kosovo        57.6   
6        Match 7                  Brazil                 Morocco        57.2   
7        Match 8                   Qatar             Switzerland        21.2   
8        Match 9             Ivory Coast                 Ecuador        47.6   
9       Match 10                 Germany                 Curaçao        78.6   
10      Match 11             Netherlands                   Japan        62.0   
11      Match 12                 Albania

In [73]:
# Save all matches to a new CSV file in your data folder
prediction_output.to_csv("data/predictions_2026_output.csv", index=False)
print("Successfully saved all predictions to data/predictions_2026_output.csv!")

Successfully saved all predictions to data/predictions_2026_output.csv!


In [72]:
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

# 1. Predict probabilities using your 4-feature array
probs = model.predict_proba(X_future_array)

# 2. Map percentages
schedule["Home Win %"] = (probs[:, 0] * 100).round(1)
schedule["Away Win %"] = (probs[:, 1] * 100).round(1)
schedule["Draw %"] = (probs[:, 2] * 100).round(1)

# 3. Create the clean output dataframe
prediction_output = schedule[["match_number", "Home Team", "Away Team", "Home Win %", "Away Win %", "Draw %"]]

# --- NEW: Tell pandas to show ALL rows ---
import pandas as pd
pd.set_option('display.max_rows', None) 

# Print the entire table
print(prediction_output)

    match_number               Home Team               Away Team  Home Win %  \
0        Match 1                  Mexico            South Africa        73.5   
1        Match 2             South Korea          Czech Republic        55.4   
2        Match 3                  Canada  Bosnia and Herzegovina        61.0   
3        Match 4           United States                Paraguay        65.5   
4        Match 5                   Haiti                Scotland        13.7   
5        Match 6               Australia                  Kosovo        57.6   
6        Match 7                  Brazil                 Morocco        57.2   
7        Match 8                   Qatar             Switzerland        21.2   
8        Match 9             Ivory Coast                 Ecuador        47.6   
9       Match 10                 Germany                 Curaçao        78.6   
10      Match 11             Netherlands                   Japan        62.0   
11      Match 12                 Albania

In [71]:
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

# Your prediction code below...
probs = model.predict_proba(X_future_array)

In [70]:
# 1. Pass the 4-feature matrix into your trained model
probs = model.predict_proba(X_future_array)

# 2. Assign the columns back to your schedule table
# Standard mapping used in your training: 0 = Home Win, 1 = Away Win, 2 = Draw
schedule["Home Win %"] = (probs[:, 0] * 100).round(1)
schedule["Away Win %"] = (probs[:, 1] * 100).round(1)
schedule["Draw %"] = (probs[:, 2] * 100).round(1)

# 3. Print your completed clean 2026 Prediction Table!
prediction_output = schedule[["match_number", "Home Team", "Away Team", "Home Win %", "Away Win %", "Draw %"]]
print(prediction_output.head(10))

  match_number      Home Team               Away Team  Home Win %  Away Win %  \
0      Match 1         Mexico            South Africa        73.5        11.3   
1      Match 2    South Korea          Czech Republic        55.4        21.2   
2      Match 3         Canada  Bosnia and Herzegovina        61.0        17.3   
3      Match 4  United States                Paraguay        65.5        16.2   
4      Match 5          Haiti                Scotland        13.7        49.1   
5      Match 6      Australia                  Kosovo        57.6        21.5   
6      Match 7         Brazil                 Morocco        57.2        26.6   
7      Match 8          Qatar             Switzerland        21.2        50.8   
8      Match 9    Ivory Coast                 Ecuador        47.6        28.4   
9     Match 10        Germany                 Curaçao        78.6         7.2   

   Draw %  
0    15.2  
1    23.4  
2    21.7  
3    18.3  
4    37.2  
5    21.0  
6    16.2  
7    27.9  


c:\Users\rites\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(


In [69]:
# 1. Create the Difference columns in the main schedule dataframe for your records
schedule["Rank Difference"] = schedule["Away Rank"] - schedule["Home Rank"]
schedule["Rating Difference"] = schedule["Home Rating"] - schedule["Away Rating"]

# 2. Extract EXACTLY the 4 baseline features that your model was fitted on
X_future = schedule[[
    "Home Rank", 
    "Away Rank", 
    "Home Rating", 
    "Away Rating"
]]

# 3. Convert to a raw matrix array to ensure flawless delivery
X_future_array = X_future.values

In [68]:
# Pass the raw numpy array to prevent any ValueError
probs = model.predict_proba(X_future_array)

# Standard mapping: 0 = Home Win, 1 = Away Win, 2 = Draw
schedule["Home Win %"] = (probs[:, 0] * 100).round(1)
schedule["Away Win %"] = (probs[:, 1] * 100).round(1)
schedule["Draw %"] = (probs[:, 2] * 100).round(1)

# Display your completed 2026 Prediction Table!
prediction_output = schedule[["match_number", "Home Team", "Away Team", "Home Win %", "Away Win %", "Draw %"]]
print(prediction_output.head(10))

c:\Users\rites\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(


ValueError: X has 6 features, but LogisticRegression is expecting 4 features as input.

In [67]:
# Create the columns both ways just to be safe
schedule["Rank Difference"] = schedule["Away Rank"] - schedule["Home Rank"]
schedule["Rating Difference"] = schedule["Home Rating"] - schedule["Away Rating"]

schedule["Rank_Difference"] = schedule["Rank Difference"]
schedule["Rating_Difference"] = schedule["Rating Difference"]

# Extract exactly the 6 columns in the precise order your model expects
X_future = schedule[[
    "Home Rank", "Away Rank", "Home Rating", "Away Rating", 
    "Rank Difference", "Rating Difference"
]]

# Convert to a raw matrix to completely bypass Scikit-Learn's feature name check
X_future_array = X_future.values

In [66]:
# Create the columns both ways just to be safe
schedule["Rank Difference"] = schedule["Away Rank"] - schedule["Home Rank"]
schedule["Rating Difference"] = schedule["Home Rating"] - schedule["Away Rating"]

schedule["Rank_Difference"] = schedule["Rank Difference"]
schedule["Rating_Difference"] = schedule["Rating Difference"]

# Extract exactly the 6 columns in the precise order your model expects
X_future = schedule[[
    "Home Rank", "Away Rank", "Home Rating", "Away Rating", 
    "Rank Difference", "Rating Difference"
]]

# Convert to a raw matrix to completely bypass Scikit-Learn's feature name check
X_future_array = X_future.values

In [65]:
# Predict win/draw/loss distribution using your trained model
probs = model.predict_proba(X_future)

# Standard mapping: 0 = Home Win, 1 = Away Win, 2 = Draw
schedule["Home Win %"] = (probs[:, 0] * 100).round(1)
schedule["Away Win %"] = (probs[:, 1] * 100).round(1)
schedule["Draw %"] = (probs[:, 2] * 100).round(1)

# Review the finalized predictions output
prediction_output = schedule[["match_number", "Home Team", "Away Team", "Home Win %", "Away Win %", "Draw %"]]
print(prediction_output.head(10))

ValueError: The feature names should match those that were passed during fit.
Feature names unseen at fit time:
- Rank_Difference
- Rating_Difference


In [64]:
# Replicate training formulas exactly using underscores
schedule["Rank_Difference"] = schedule["Away Rank"] - schedule["Home Rank"]
schedule["Rating_Difference"] = schedule["Home Rating"] - schedule["Away Rating"]

# Select the feature array with matching column names
X_future = schedule[[
    "Home Rank", "Away Rank", "Home Rating", "Away Rating", 
    "Rank_Difference", "Rating_Difference"
]]

In [63]:
# Predict win/draw/loss distribution using your best model
# Classes mapping check: 0 = Home Win, 1 = Away Win, 2 = Draw
probs = model.predict_proba(X_future)

schedule["Home Win %"] = (probs[:, 0] * 100).round(1)
schedule["Away Win %"] = (probs[:, 1] * 100).round(1)
schedule["Draw %"] = (probs[:, 2] * 100).round(1)

# Review final outputs
prediction_output = schedule[["match_number", "Home Team", "Away Team", "Home Win %", "Away Win %", "Draw %"]]
print(prediction_output.head(10))

ValueError: The feature names should match those that were passed during fit.
Feature names unseen at fit time:
- Rank_Difference
- Rating_Difference


In [60]:
# Replicate training formulas exactly
schedule["Rank Difference"] = schedule["Away Rank"] - schedule["Home Rank"]
schedule["Rating Difference"] = schedule["Home Rating"] - schedule["Away Rating"]

# Select the feature array 
X_future = schedule[[
    "Home Rank", "Away Rank", "Home Rating", "Away Rating", 
    "Rank Difference", "Rating Difference"
]]

In [59]:
# Merge Home Team rankings
schedule = schedule.merge(
    ranking[["Country", "Rank", "Rating"]],
    left_on="Home Team",
    right_on="Country",
    how="left"
)
schedule.rename(columns={"Rank": "Home Rank", "Rating": "Home Rating"}, inplace=True)
schedule.drop(columns=["Country"], inplace=True)

# Merge Away Team rankings
schedule = schedule.merge(
    ranking[["Country", "Rank", "Rating"]],
    left_on="Away Team",
    right_on="Country",
    how="left"
)
schedule.rename(columns={"Rank": "Away Rank", "Rating": "Away Rating"}, inplace=True)
schedule.drop(columns=["Country"], inplace=True)

# Fill any missing placeholder rankings with a conservative average if necessary
schedule[["Home Rank", "Away Rank"]] = schedule[["Home Rank", "Away Rank"]].fillna(50)
schedule[["Home Rating", "Away Rating"]] = schedule[["Home Rating", "Away Rating"]].fillna(1500)

In [58]:
# Split combined team names on the ' v ' delimiter
schedule[['Home Team', 'Away Team']] = schedule['teams'].str.split(' v ', expand=True)

# Clean up long placeholder group names for non-determined teams if any remain
schedule['Home Team'] = schedule['Home Team'].str.split('/').str[0].str.strip()
schedule['Away Team'] = schedule['Away Team'].str.split('/').str[0].str.strip()

# Apply the same country map to the 2026 schedule to avoid merge issues
schedule["Home Team"] = schedule["Home Team"].replace(country_map)
schedule["Away Team"] = schedule["Away Team"].replace(country_map)

In [57]:
schedule.head()

,date,match_number,teams,group,stadium,date_dt
0,"Thursday, 11 June 2026",Match 1,Mexico v South Africa,Group A,Mexico City Stadium,2026-06-11
1,"Thursday, 11 June 2026",Match 2,Korea Republic v Czechia/Denmark/North Macedon...,Group A,Estadio Guadalajara,2026-06-11
2,"Friday, 12 June 2026",Match 3,Canada v Bosnia and Herzegovina/Italy/Northern...,Group B,Toronto Stadium,2026-06-12
3,"Friday, 12 June 2026",Match 4,USA v Paraguay,Group D,Los Angeles Stadium,2026-06-12
4,"Saturday, 13 June 2026",Match 5,Haiti v Scotland,Group C,Boston Stadium,2026-06-13


In [56]:
schedule.columns

Index(['date', 'match_number', 'teams', 'group', 'stadium', 'date_dt'], dtype='object')

In [55]:
print("Accuracy :", accuracy_score(y_test, rf_prediction))

print("Precision :", precision_score(y_test, rf_prediction, average="macro", zero_division=0))

print("Recall :", recall_score(y_test, rf_prediction, average="macro", zero_division=0))

print("F1 Score :", f1_score(y_test, rf_prediction, average="macro", zero_division=0))

Accuracy : 0.41304347826086957
Precision : 0.35141093474426804
Recall : 0.3680555555555556
F1 Score : 0.3569992099403865


In [54]:
rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf.fit(X_train, y_train)

rf_prediction = rf.predict(X_test)

In [53]:
print(X_train.columns)

Index(['Home Rank', 'Away Rank', 'Home Rating', 'Away Rating',
       'Rank Difference', 'Rating Difference'],
      dtype='object')


In [52]:
X_train = train[features]
X_test = test[features]

y_train = train["Result"]
y_test = test["Result"]

In [51]:
features = [
    "Home Rank",
    "Away Rank",
    "Home Rating",
    "Away Rating",
    "Rank Difference",
    "Rating Difference"
]

In [50]:
train = matches[matches["Year"] <= 2018]

test = matches[matches["Year"] == 2022]

In [49]:
X_train = train[features]

X_test = test[features]

KeyError: "['Rank Difference', 'Rating Difference'] not in index"

In [48]:
features = [
    "Home Rank",
    "Away Rank",
    "Home Rating",
    "Away Rating",
    "Rank Difference",
    "Rating Difference"
]

In [46]:
matches[[
    "Home Rank",
    "Away Rank",
    "Rank Difference",
    "Home Rating",
    "Away Rating",
    "Rating Difference"
]].head()

,Home Rank,Away Rank,Rank Difference,Home Rating,Away Rating,Rating Difference
0,2.0,9.0,7.0,1906.84,1736.01,170.83
1,15.0,10.0,-5.0,1677.17,1735.41,-58.24
4,1.0,2.0,1.0,1907.40,1906.84,0.56
7,15.0,37.0,22.0,1677.17,1520.59,156.58
10,1.0,9.0,8.0,1907.40,1736.01,171.39


In [45]:
matches["Rank Difference"] = matches["Away Rank"] - matches["Home Rank"]

matches["Rating Difference"] = matches["Home Rating"] - matches["Away Rating"]

In [44]:
print("Accuracy :", accuracy_score(y_test, prediction))

print("Precision :", precision_score(y_test, prediction, average="macro", zero_division=0))

print("Recall :", recall_score(y_test, prediction, average="macro", zero_division=0))

print("F1 Score :", f1_score(y_test, prediction, average="macro", zero_division=0))

Accuracy : 0.5869565217391305
Precision : 0.41141141141141135
Recall : 0.4166666666666667
F1 Score : 0.38950819672131143


In [43]:
print("Accuracy :", accuracy_score(y_test, prediction))

print("Precision :", precision_score(y_test, prediction, average="macro"))

print("Recall :", recall_score(y_test, prediction, average="macro"))

print("F1 Score :", f1_score(y_test, prediction, average="macro"))

Accuracy : 0.5869565217391305
Precision : 0.41141141141141135
Recall : 0.4166666666666667
F1 Score : 0.38950819672131143


c:\Users\rites\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [42]:
model = LogisticRegression(max_iter=1000)

model.fit(X_train, y_train)

prediction = model.predict(X_test)

## Logistic Regression

In [41]:
print(X_train.shape)
print(X_test.shape)

print(y_train.shape)
print(y_test.shape)

(374, 4)
(46, 4)
(374,)
(46,)


In [40]:
features = [
    "Home Rank",
    "Away Rank",
    "Home Rating",
    "Away Rating"
]

X_train = train[features]
X_test = test[features]

y_train = train["Result"]
y_test = test["Result"]

In [39]:
print("Training data:", train.shape)

print("Testing data:", test.shape)

Training data: (374, 42)
Testing data: (46, 42)


In [38]:
train = matches[matches["Year"] <= 2018]

test = matches[matches["Year"] == 2022]

In [37]:
sorted(matches["Year"].unique())

[np.int32(1930),
 np.int32(1934),
 np.int32(1938),
 np.int32(1950),
 np.int32(1954),
 np.int32(1958),
 np.int32(1962),
 np.int32(1966),
 np.int32(1970),
 np.int32(1974),
 np.int32(1978),
 np.int32(1982),
 np.int32(1986),
 np.int32(1990),
 np.int32(1994),
 np.int32(1998),
 np.int32(2002),
 np.int32(2006),
 np.int32(2010),
 np.int32(2014),
 np.int32(2018),
 np.int32(2022)]

In [36]:
matches[["Match Date", "Year"]].head()

,Match Date,Year
0,7/13/1930,1930
1,7/13/1930,1930
4,7/15/1930,1930
7,7/17/1930,1930
10,7/19/1930,1930


In [35]:
matches["Year"] = pd.to_datetime(matches["Match Date"]).dt.year

In [34]:
matches["Match Date"].head(10)

0     7/13/1930
1     7/13/1930
4     7/15/1930
7     7/17/1930
10    7/19/1930
12    7/20/1930
15    7/26/1930
17    7/30/1930
18    5/27/1934
20    5/27/1934
Name: Match Date, dtype: object

In [33]:
y.head()

0     0
1     0
4     0
7     0
10    0
Name: Result, dtype: int64

In [32]:
X.head()

,Home Rank,Away Rank,Home Rating,Away Rating
0,2.0,9.0,1906.84,1736.01
1,15.0,10.0,1677.17,1735.41
4,1.0,2.0,1907.40,1906.84
7,15.0,37.0,1677.17,1520.59
10,1.0,9.0,1907.40,1736.01


In [31]:
features = [
    "Home Rank",
    "Away Rank",
    "Home Rating",
    "Away Rating"
]

X = matches[features]

y = matches["Result"]

In [30]:
matches.columns

Index(['Key Id', 'Tournament Id', 'tournament Name', 'Match Id', 'Match Name',
       'Stage Name', 'Group Name', 'Group Stage', 'Knockout Stage', 'Replayed',
       'Replay', 'Match Date', 'Match Time', 'Stadium Id', 'Stadium Name',
       'City Name', 'Country Name', 'Home Team Id', 'Home Team Name',
       'Home Team Code', 'Away Team Id', 'Away Team Name', 'Away Team Code',
       'Score', 'Home Team Score', 'Away Team Score', 'Home Team Score Margin',
       'Away Team Score Margin', 'Extra Time', 'Penalty Shootout',
       'Score Penalties', 'Home Team Score Penalties',
       'Away Team Score Penalties', 'Result', 'Home Team Win', 'Away Team Win',
       'Draw', 'Home Rank', 'Home Rating', 'Away Rank', 'Away Rating'],
      dtype='object')

In [29]:
matches["Result"].value_counts()

Result
0    227
1    119
2     74
Name: count, dtype: int64

In [28]:
matches["Result"] = matches["Result"].replace({
    "home team win": 0,
    "away team win": 1,
    "draw": 2
})

matches["Result"] = matches["Result"].astype(int)

C:\Users\rites\AppData\Local\Temp\ipykernel_15812\1912086891.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  matches["Result"] = matches["Result"].replace({


In [27]:
matches["Result"].value_counts()

Result
home team win    227
away team win    119
draw              74
Name: count, dtype: int64

In [26]:
print(matches.shape)

(420, 41)


In [25]:
matches = matches.dropna(subset=["Home Rank", "Away Rank"])

In [24]:
print("Missing Home Rank:", matches["Home Rank"].isnull().sum())

print("Missing Away Rank:", matches["Away Rank"].isnull().sum())

Missing Home Rank: 316
Missing Away Rank: 339


In [23]:
matches[[
    "Home Team Name",
    "Home Rank",
    "Away Team Name",
    "Away Rank"
]].head(15)

,Home Team Name,Home Rank,Away Team Name,Away Rank
0,France,2.0,Mexico,9.0
1,United States,15.0,Belgium,10.0
2,Yugoslavia,NaN,Brazil,5.0
3,Romania,NaN,Peru,NaN
4,Argentina,1.0,France,2.0
5,Chile,NaN,Mexico,9.0
6,Yugoslavia,NaN,Bolivia,NaN
7,United States,15.0,Paraguay,37.0
8,Uruguay,19.0,Peru,NaN
9,Chile,NaN,France,2.0


In [22]:
matches = matches.merge(
    ranking[["Country", "Rank", "Rating"]],
    left_on="Away Team Name",
    right_on="Country",
    how="left"
)

matches.rename(columns={
    "Rank": "Away Rank",
    "Rating": "Away Rating"
}, inplace=True)

matches.drop(columns=["Country"], inplace=True)

In [21]:
matches = matches.merge(
    ranking[["Country", "Rank", "Rating"]],
    left_on="Home Team Name",
    right_on="Country",
    how="left"
)

matches.rename(columns={
    "Rank": "Home Rank",
    "Rating": "Home Rating"
}, inplace=True)

matches.drop(columns=["Country"], inplace=True)

In [20]:
country_map = {
    "USA": "United States",
    "IR Iran": "Iran",
    "Korea Republic": "South Korea",
    "Türkiye": "Turkey",
    "Côte d'Ivoire": "Ivory Coast",
    "Congo DR": "DR Congo",
    "Czechia": "Czech Republic"
}

ranking["Country"] = ranking["Country"].replace(country_map)

In [19]:
match_countries = set(matches["Home Team Name"]) | set(matches["Away Team Name"])
ranking_countries = set(ranking["Country"])

print("Countries in matches but not in rankings:")
print(sorted(match_countries - ranking_countries))

print("\nCountries in rankings but not in matches:")
print(sorted(ranking_countries - match_countries))

Countries in matches but not in rankings:
['Angola', 'Bolivia', 'Bulgaria', 'Cameroon', 'Chile', 'China', 'Costa Rica', 'Cuba', 'Czech Republic', 'Czechoslovakia', 'Denmark', 'Dutch East Indies', 'East Germany', 'El Salvador', 'Greece', 'Honduras', 'Hungary', 'Iceland', 'Iran', 'Israel', 'Italy', 'Ivory Coast', 'Jamaica', 'Kuwait', 'Nigeria', 'North Korea', 'Northern Ireland', 'Peru', 'Poland', 'Republic of Ireland', 'Romania', 'Russia', 'Serbia', 'Serbia and Montenegro', 'Slovakia', 'Slovenia', 'South Korea', 'Soviet Union', 'Togo', 'Trinidad and Tobago', 'Turkey', 'Ukraine', 'United Arab Emirates', 'United States', 'Wales', 'West Germany', 'Yugoslavia', 'Zaire']

Countries in rankings but not in matches:
['Cabo Verde', 'Congo DR', 'Curaçao', 'Czechia', "Côte d'Ivoire", 'IR Iran', 'Jordan', 'Korea Republic', 'Türkiye', 'USA', 'Uzbekistan']


In [18]:
sorted(matches["Home Team Name"].unique())

['Algeria',
 'Angola',
 'Argentina',
 'Australia',
 'Austria',
 'Belgium',
 'Bolivia',
 'Bosnia and Herzegovina',
 'Brazil',
 'Bulgaria',
 'Cameroon',
 'Canada',
 'Chile',
 'China',
 'Colombia',
 'Costa Rica',
 'Croatia',
 'Cuba',
 'Czech Republic',
 'Czechoslovakia',
 'Denmark',
 'East Germany',
 'Ecuador',
 'Egypt',
 'England',
 'France',
 'Germany',
 'Ghana',
 'Greece',
 'Haiti',
 'Honduras',
 'Hungary',
 'Iceland',
 'Iran',
 'Iraq',
 'Italy',
 'Ivory Coast',
 'Jamaica',
 'Japan',
 'Mexico',
 'Morocco',
 'Netherlands',
 'New Zealand',
 'Nigeria',
 'North Korea',
 'Northern Ireland',
 'Norway',
 'Panama',
 'Paraguay',
 'Peru',
 'Poland',
 'Portugal',
 'Qatar',
 'Republic of Ireland',
 'Romania',
 'Russia',
 'Saudi Arabia',
 'Scotland',
 'Senegal',
 'Serbia',
 'Serbia and Montenegro',
 'Slovakia',
 'Slovenia',
 'South Africa',
 'South Korea',
 'Soviet Union',
 'Spain',
 'Sweden',
 'Switzerland',
 'Togo',
 'Trinidad and Tobago',
 'Tunisia',
 'Turkey',
 'Ukraine',
 'United Arab Emirates

In [17]:
matches["Away Team Name"].unique()

array(['Mexico', 'Belgium', 'Brazil', 'Peru', 'France', 'Bolivia',
       'Paraguay', 'Romania', 'Chile', 'United States', 'Yugoslavia',
       'Argentina', 'Egypt', 'Netherlands', 'Hungary', 'Switzerland',
       'Sweden', 'Spain', 'Germany', 'Austria', 'Czechoslovakia',
       'Dutch East Indies', 'Norway', 'Poland', 'Cuba', 'Italy',
       'England', 'Scotland', 'South Korea', 'Turkey', 'West Germany',
       'Uruguay', 'Wales', 'Northern Ireland', 'Soviet Union', 'Colombia',
       'Bulgaria', 'North Korea', 'Portugal', 'Israel', 'El Salvador',
       'Morocco', 'Australia', 'Haiti', 'East Germany', 'Zaire', 'Iran',
       'Tunisia', 'Cameroon', 'New Zealand', 'Algeria', 'Honduras',
       'Kuwait', 'Iraq', 'Denmark', 'Canada', 'Republic of Ireland',
       'United Arab Emirates', 'Costa Rica', 'Russia', 'Saudi Arabia',
       'Greece', 'Nigeria', 'South Africa', 'Japan', 'Croatia', 'Jamaica',
       'Senegal', 'Slovenia', 'Ecuador', 'China', 'Ivory Coast',
       'Czech Republic',

In [16]:
matches["Home Team Name"].unique()

array(['France', 'United States', 'Yugoslavia', 'Romania', 'Argentina',
       'Chile', 'Uruguay', 'Brazil', 'Paraguay', 'Austria',
       'Czechoslovakia', 'Germany', 'Hungary', 'Italy', 'Spain', 'Sweden',
       'Switzerland', 'Cuba', 'England', 'West Germany', 'Turkey',
       'Northern Ireland', 'Soviet Union', 'Mexico', 'Wales', 'Portugal',
       'North Korea', 'Peru', 'Belgium', 'Bulgaria', 'East Germany',
       'Zaire', 'Poland', 'Australia', 'Scotland', 'Netherlands', 'Haiti',
       'Tunisia', 'Algeria', 'Honduras', 'Canada', 'Morocco',
       'South Korea', 'Iraq', 'Denmark', 'United Arab Emirates',
       'Costa Rica', 'Cameroon', 'Republic of Ireland', 'Colombia',
       'Norway', 'Nigeria', 'Saudi Arabia', 'Bolivia', 'Russia', 'Greece',
       'Jamaica', 'South Africa', 'Japan', 'Croatia', 'China', 'Senegal',
       'Slovenia', 'Ecuador', 'Trinidad and Tobago',
       'Serbia and Montenegro', 'Angola', 'Czech Republic', 'Togo',
       'Iran', 'Ivory Coast', 'Ghana', 'Ukr

In [15]:
ranking.columns

Index(['Rank', 'Country', 'Rating'], dtype='object')

In [14]:
ranking.head()

,Rank,Country,Rating
0,1,Argentina,1907.40
1,2,France,1906.84
2,3,Spain,1879.58
3,4,England,1840.46
4,5,Brazil,1785.19


In [13]:
matches.isnull().sum()

Key Id                       0
Tournament Id                0
tournament Name              0
Match Id                     0
Match Name                   0
Stage Name                   0
Group Name                   0
Group Stage                  0
Knockout Stage               0
Replayed                     0
Replay                       0
Match Date                   0
Match Time                   0
Stadium Id                   0
Stadium Name                 0
City Name                    0
Country Name                 0
Home Team Id                 0
Home Team Name               0
Home Team Code               0
Away Team Id                 0
Away Team Name               0
Away Team Code               0
Score                        0
Home Team Score              0
Away Team Score              0
Home Team Score Margin       0
Away Team Score Margin       0
Extra Time                   0
Penalty Shootout             0
Score Penalties              0
Home Team Score Penalties    0
Away Tea

In [12]:
matches.describe()

,Key Id,Group Stage,Knockout Stage,Replayed,Replay,Home Team Score,Away Team Score,Home Team Score Margin,Away Team Score Margin,Extra Time,Penalty Shootout,Home Team Score Penalties,Away Team Score Penalties,Home Team Win,Away Team Win,Draw
count,964.00000,964.000000,964.000000,964.000000,964.000000,964.000000,964.000000,964.000000,964.000000,964.000000,964.000000,964.000000,964.000000,964.000000,964.000000,964.000000
mean,482.50000,0.744813,0.255187,0.004149,0.004149,1.766598,1.054979,0.711618,-0.711618,0.075726,0.036307,0.121369,0.108921,0.565353,0.248963,0.185685
std,278.42713,0.436192,0.436192,0.064315,0.064315,1.601040,1.071720,1.925893,1.925893,0.264697,0.187150,0.665731,0.600764,0.495968,0.432637,0.389054
min,1.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-7.000000,-9.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,241.75000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,-2.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,482.50000,1.000000,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000,-1.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000
75%,723.25000,1.000000,1.000000,0.000000,0.000000,3.000000,2.000000,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000
max,964.00000,1.000000,1.000000,1.000000,1.000000,10.000000,7.000000,9.000000,7.000000,1.000000,1.000000,5.000000,5.000000,1.000000,1.000000,1.000000


In [11]:
matches.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 964 entries, 0 to 963
Data columns (total 37 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   Key Id                     964 non-null    int64 
 1   Tournament Id              964 non-null    object
 2   tournament Name            964 non-null    object
 3   Match Id                   964 non-null    object
 4   Match Name                 964 non-null    object
 5   Stage Name                 964 non-null    object
 6   Group Name                 964 non-null    object
 7   Group Stage                964 non-null    int64 
 8   Knockout Stage             964 non-null    int64 
 9   Replayed                   964 non-null    int64 
 10  Replay                     964 non-null    int64 
 11  Match Date                 964 non-null    object
 12  Match Time                 964 non-null    object
 13  Stadium Id                 964 non-null    object
 14  Stadium Na

## Dataset Information

In [10]:
schedule.columns

Index(['date', 'match_number', 'teams', 'group', 'stadium', 'date_dt'], dtype='object')

In [9]:
ranking.columns

Index(['Rank', 'Country', 'Rating'], dtype='object')

In [8]:
matches.columns

Index(['Key Id', 'Tournament Id', 'tournament Name', 'Match Id', 'Match Name',
       'Stage Name', 'Group Name', 'Group Stage', 'Knockout Stage', 'Replayed',
       'Replay', 'Match Date', 'Match Time', 'Stadium Id', 'Stadium Name',
       'City Name', 'Country Name', 'Home Team Id', 'Home Team Name',
       'Home Team Code', 'Away Team Id', 'Away Team Name', 'Away Team Code',
       'Score', 'Home Team Score', 'Away Team Score', 'Home Team Score Margin',
       'Away Team Score Margin', 'Extra Time', 'Penalty Shootout',
       'Score Penalties', 'Home Team Score Penalties',
       'Away Team Score Penalties', 'Result', 'Home Team Win', 'Away Team Win',
       'Draw'],
      dtype='object')

In [7]:
schedule.head()

,date,match_number,teams,group,stadium,date_dt
0,"Thursday, 11 June 2026",Match 1,Mexico v South Africa,Group A,Mexico City Stadium,2026-06-11
1,"Thursday, 11 June 2026",Match 2,Korea Republic v Czechia/Denmark/North Macedon...,Group A,Estadio Guadalajara,2026-06-11
2,"Friday, 12 June 2026",Match 3,Canada v Bosnia and Herzegovina/Italy/Northern...,Group B,Toronto Stadium,2026-06-12
3,"Friday, 12 June 2026",Match 4,USA v Paraguay,Group D,Los Angeles Stadium,2026-06-12
4,"Saturday, 13 June 2026",Match 5,Haiti v Scotland,Group C,Boston Stadium,2026-06-13


In [6]:
ranking.head()

,Rank,Country,Rating
0,1,Argentina,1907.40
1,2,France,1906.84
2,3,Spain,1879.58
3,4,England,1840.46
4,5,Brazil,1785.19


In [5]:
matches.head()

,Key Id,Tournament Id,tournament Name,Match Id,Match Name,Stage Name,Group Name,Group Stage,Knockout Stage,Replayed,...,Away Team Score Margin,Extra Time,Penalty Shootout,Score Penalties,Home Team Score Penalties,Away Team Score Penalties,Result,Home Team Win,Away Team Win,Draw
0,1,WC-1930,1930 FIFA World Cup,M-1930-01,France v Mexico,group stage,Group 1,1,0,0,...,-3,0,0,0-0,0,0,home team win,1,0,0
1,2,WC-1930,1930 FIFA World Cup,M-1930-02,United States v Belgium,group stage,Group 4,1,0,0,...,-3,0,0,0-0,0,0,home team win,1,0,0
2,3,WC-1930,1930 FIFA World Cup,M-1930-03,Yugoslavia v Brazil,group stage,Group 2,1,0,0,...,-1,0,0,0-0,0,0,home team win,1,0,0
3,4,WC-1930,1930 FIFA World Cup,M-1930-04,Romania v Peru,group stage,Group 3,1,0,0,...,-2,0,0,0-0,0,0,home team win,1,0,0
4,5,WC-1930,1930 FIFA World Cup,M-1930-05,Argentina v France,group stage,Group 1,1,0,0,...,-1,0,0,0-0,0,0,home team win,1,0,0


In [4]:
matches = pd.read_csv(
    "data/FIFA World Cup 1930-2022 All Match Dataset.csv",
    encoding="cp1252"
)
ranking = pd.read_csv("data/fifa_rankings.csv")

schedule = pd.read_csv("data/schedule_2026.csv")

## Load Dataset

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import mean_absolute_error

# FIFA World Cup Prediction

## Round 2 - FIFA World Cup Prediction

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns